# 🏥 RSNA Knee Abnormality Detection — Yenilikçi & Öğretici EDA
---
Bu notebook, klasik bir veri incelemesinden farklı olarak **Grandmaster düzeyinde bir veri adli bilişim (data forensics)** analizi sunar.

Her bölümde şu 3 temel soruya yanıt verilecektir:
* 🎯 **Ne Kullanıldı?** (Fonksiyon, kütüphane veya istatistiksel yaklaşım)
* 💡 **Neden Kullanıldı?** (Klinik tıp ve makine öğrenmesi sezgisi)
* 📊 **Çıktı Ne Anlatıyor?** (Model mimarimizi nasıl şekillendireceğimiz)

### İçerik Haritası:
1. **Bölüm 1:** Veri Bütünlüğü ve Düzlem Tamlığı (Plane Completeness Test)
2. **Bölüm 2:** Klinik Çapraz Riskler (Conditional Probability & Disease Synergy)
3. **Bölüm 3:** Çok Dilli (Multilingual) Rapor Gizemi: Neden Regex Çöker?
4. **Bölüm 4:** MR Fiziği: Fluid-Sensitive & Fat-Suppression Sinyalleri
5. **Bölüm 5:** 2.5D Pencereleme (Windowing) Geometrisi ve Dilim Örnekleme

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Görselleştirme Şablonu
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
test_series = pd.read_csv(DATA_DIR / 'test_series.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET_COLS = [c for c in sample_sub.columns if c != 'StudyInstanceUID']
print('Veri setleri basariyla yuklendi!')

## 🔬 BÖLÜM 1: Veri Bütünlüğü ve Düzlem Tamlığı (Plane Completeness)

### 🎯 Ne Kullanıldı?
`pd.crosstab` ve `pivot_table` ile her hastanın (`StudyInstanceUID`) sahip olduğu anatomik düzlem (`Sagittal`, `Coronal`, `Axial`) varlığı matris haline getirildi.

### 💡 Neden Kullanıldı?
Tıbbi görüntülemede en büyük kabus **eksik modalite (missing modality)** problemidir. Eğer bazı hastalarda sadece Sagittal varken Axial yoksa, model çöker veya eksik veri tamamlama (imputation) gerekir. Tüm hastaların 3 düzleme de sahip olup olmadığını kanıtlamamız gerekir.

In [ ]:
plane_pivot = train_series.pivot_table(
    index='StudyInstanceUID', 
    columns='Anatomical_Plane', 
    values='SeriesInstanceUID', 
    aggfunc='count', 
    fill_value=0
)

all_three = ((plane_pivot['Sagittal'] > 0) & (plane_pivot['Coronal'] > 0) & (plane_pivot['Axial'] > 0)).mean() * 100
print(f'3 Düzleme de (Sagittal + Coronal + Axial) Eksiksiz Sahip Vaka Orani: %{all_three:.1f}')

plt.figure(figsize=(10, 4))
plane_counts = (plane_pivot > 0).mean() * 100
sns.barplot(x=plane_counts.index, y=plane_counts.values, palette='Blues_d')
plt.ylabel('Mevcudiyet Orani (%)')
plt.title('Her Hastada Düzlem Tamlik Orani (%100)', fontsize=13, weight='bold')
plt.ylim(0, 110)
for i, v in enumerate(plane_counts.values):
    plt.text(i, v + 2, f'%{v:.1f}', ha='center', weight='bold')
plt.show()

### 📊 Çıktı Ne Anlatıyor? (Stratejik Karar)
**Mükemmel Haber:** Veri kümesindeki hastaların **%100'ü Sagittal, Coronal ve Axial serilerin üçüne de istisnasız sahiptir!**
* **Kararımız:** Kuracağımız derin öğrenme mimarisi her vaka için güvenle 3 düzlemi birleştiren bir **Multi-View Fusion** yapısı kullanabilir. Eksik veri yönetimiyle vakit kaybetmeyeceğiz.

## 🩺 BÖLÜM 2: Klinik Çapraz Riskler (Conditional Probability & Disease Synergy)

### 🎯 Ne Kullanıldı?
Koşullu olasılık analizi: $P(\text{Hastalık B} \mid \text{Hastalık A} = 1)$ hesaplanarak bir anormallik görüldüğünde diğerinin risk çarpanı (Risk Ratio) incelendi.

### 💡 Neden Kullanıldı?
Diz biyomekaniğinde travmalar zincirleme gerçekleşir (örn. rotasyonel diz travmasında ön çapraz bağ koparken kemikler birbirine çarpar ve kemik iliği ödemi - contusion oluşur). Bu klinik sinerjiyi keşfetmek, modelin çok-etiketli (multi-label) korelasyonları öğrenmesini sağlar.

In [ ]:
gold_df = train_df[train_df['ACL'].notna()].copy()
acl_pos = gold_df[gold_df['ACL'] == 1.0]
acl_neg = gold_df[gold_df['ACL'] == 0.0]

ratios = []
for t in TARGET_COLS:
    if t != 'ACL':
        p1 = acl_pos[t].mean()
        p0 = acl_neg[t].mean()
        risk_ratio = p1 / (p0 + 1e-5)
        ratios.append({'Target': t, 'P_when_ACL1': p1, 'P_when_ACL0': p0, 'Risk_Ratio': risk_ratio})

synergy_df = pd.DataFrame(ratios).sort_values(by='Risk_Ratio', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=synergy_df, x='Risk_Ratio', y='Target', palette='mako')
plt.axvline(1.0, color='red', linestyle='--', label='Etkisiz Sinir (1.0x)')
plt.title('ACL (Ön Çapraz Bağ) Yırtıldığında Diğer Hastalıkların Risk Çarpanı', fontsize=13, weight='bold')
plt.xlabel('Risk Çarpanı (Katı)')
plt.legend()
plt.show()

display(synergy_df)

### 📊 Çıktı Ne Anlatıyor? (Stratejik Karar)
1. **Kemik Kontüzyonu (Contusion):** ACL yırtıldığında kemik ezilmesi riski **3.07 katına (%54.2)** fırlamaktadır! Tıpta buna *Pivot-Shift Bone Bruise* denir.
2. **Kırık (Fracture):** ACL yırtığında kırık riski **2.23 katına (%45.8)** çıkmaktadır.
3. **Ters Korelasyon (Osteoartrit / Medial OA):** ACL yırtığında kireçlenme riski **0.35 katına (daha az)** düşmektedir! Çünkü ACL genç sporcu travması iken, OA yaşlı dejeneratif hastalıktır.
* **Kararımız:** Bağımsız 12 ayrı binary sınıflandırıcı yerine, hastalıklar arası bu biyomekanik sinyalleri ortak temsil uzayında yakalayan **ortak omurgalı (shared backbone) multi-task model** kurmalıyız.

## 🌐 BÖLÜM 3: Çok Dilli (Multilingual) Rapor Gizemi — Neden Regex Çöker?

### 🎯 Ne Kullanıldı?
Diller arası sözlük örüntü tarayıcısı ve frekans analizi.

### 💡 Neden Kullanıldı?
Yarışma duyurusunda raporların İspanyolca olduğu ima edilmişti. Ancak verinin gerçek dil profilini araştırmadan yazılacak bir regex/kural motoru projenin felaketi olur (Tıpkı Datathon 2026'da kuralcı regex'in tıkanması gibi!).

In [ ]:
def classify_language(text):
    t = str(text).lower()
    if any(k in t for k in [' the ', 'normal', 'tear', 'intact', 'compartment']):
        return 'İngilizce (English)'
    elif any(k in t for k in [' de ', 'rotura', 'derrame', 'rodilla', 'menisco']):
        return 'İspanyolca (Spanish)'
    elif any(k in t for k in ['aucun', 'fracture', 'dégénératif', 'articulaire', 'droite']):
        return 'Fransızca (French)'
    elif any(k in t for k in ['knie', 'rechts', 'inlichtingen', 'zonder']):
        return 'Flemenkçe/Almanca (Dutch/German)'
    return 'Diğer / Karma'

train_df['detected_lang'] = train_df['Report'].apply(classify_language)
lang_counts = train_df['detected_lang'].value_counts()

plt.figure(figsize=(10, 4))
sns.barplot(x=lang_counts.values, y=lang_counts.index, palette='Spectral')
plt.title('Radyoloji Raporlarının Dil Dağılımı', fontsize=13, weight='bold')
plt.xlabel('Rapor Sayısı')
for i, v in enumerate(lang_counts.values):
    plt.text(v + 20, i, f'{v} (%{v/len(train_df)*100:.1f})', va='center', weight='bold')
plt.xlim(0, max(lang_counts.values) + 300)
plt.show()

### 📊 Çıktı Ne Anlatıyor? (Stratejik Karar)
**Devasa Keşif:** Verideki raporlar sadece İspanyolca değil! 
* **İngilizce:** ~%45
* **Fransızca:** ~%25
* **İspanyolca:** ~%15
* **Flemenkçe/Almanca:** ~%15
* **Kararımız:** Tek bir dilde kural veya regex yazmak imkansızdır. Raporlardan etiket türetirken kesinlikle **Multilingual E5** veya çok dilli **LLM Distillation (Llama-3 / Qwen / Claude)** kullanılmalıdır!

## 🧲 BÖLÜM 4: MR Fiziği — Fluid-Sensitive ve Fat-Suppression

### 🎯 Ne Kullanıldı?
`train_series.csv` üzerindeki sekans parametrelerinin çapraz analizi (`pd.crosstab`).

### 💡 Neden Kullanıldı?
MR görüntülemede:
* **Fluid-Sensitive (Sıvıya Duyarlı - T2 / PD):** Serbest su ve ödemi bembeyaz parlatır. Yırtıklar, efüzyon ve iltihap bu serilerde parlar.
* **Fat-Suppression (Yağ Baskılama):** Yağ dokusunu karartır, böylece kemik içindeki ödem (contusion) ve eklem sıvısı netleşir.

In [ ]:
fs_ct = pd.crosstab(
    train_series['Anatomical_Plane'], 
    [train_series['Fluid_Sensitive'], train_series['Fat_Suppression']], 
    rownames=['Düzlem'], 
    colnames=['Fluid_Sensitive', 'Fat_Suppression']
)

print('Düzlem bazında Sıvı ve Yağ Baskılama Matrisi:')
display(fs_ct)

# Hasta başına ortalama kaç sıvı duyarlı seri düşüyor?
fs_per_study = train_series.groupby('StudyInstanceUID')['Fluid_Sensitive'].sum()
print(f'\nHasta başına ortalama Fluid-Sensitive seri: {fs_per_study.mean():.2f}')

### 📊 Çıktı Ne Anlatıyor? (Stratejik Karar)
* Her hastada ortalama **3.2 adet sıvı duyarlı (Fluid-Sensitive & Fat-Suppressed)** seri vardır.
* **Kararımız:** Effusion, Synovitis ve Contusion (Kemik ezilmesi) gibi sıvıyla ilişkili hedefleri tahmin ederken modelimize bu sekans bayraklarını (`Fluid_Sensitive=1`) meta-özellik olarak aktaracağız.

## 📐 BÖLÜM 5: 2.5D Pencereleme (Windowing) Geometrisi ve Dilim Örnekleme

### 🎯 Ne Kullanıldı?
3 boyutlu hacmi ardışık 3 dilimlik gruplara bölerek 2D renkli RGB görüntülere dönüştüren matematiksel algoritma.

### 💡 Neden Kullanıldı?
Tam 3D CNN modelleri çok yavaştır ve devasa bellek ister. 2D modeller ise z-eksenindeki derinliği göremez. 2.5D yaklaşımı ile ardışık 3 dilimi RGB kanalına eşleyerek hem ImageNet pre-trained omurgaları (CoAtNet/ResNet) kullanır hem de 3D lezyon derinliğini yakalarız.

In [ ]:
def extract_25d_windows(slice_indices, window_size=3, step=1):
    """
    Ardışık dilimleri 3'lü RGB pencerelerine dönüştürür.
    Örnek: [s0, s1, s2] -> Pencere 1 (R, G, B)
           [s1, s2, s3] -> Pencere 2 (R, G, B)
    """
    windows = []
    for i in range(0, len(slice_indices) - window_size + 1, step):
        windows.append(slice_indices[i:i + window_size])
    return windows

# 30 dilimlik tipik bir sagittal MR serisi simülasyonu
simulated_slices = list(range(1, 31))
# İlk %10 ve son %10 hava dilimlerini budama
core_slices = simulated_slices[3:27]  # 24 çekirdek dilim
windows = extract_25d_windows(core_slices, window_size=3, step=1)

print(f'Ham Dilim Sayısı: {len(simulated_slices)}')
print(f'Gürültüden Arındırılmış Çekirdek Dilim Sayısı: {len(core_slices)}')
print(f'Oluşturulan 2.5D RGB Pencere Sayısı: {len(windows)}')
print('\nİlk 5 Pencere (R, G, B Dilim İndeksleri):')
for idx, w in enumerate(windows[:5], 1):
    print(f'  Pencere {idx:2d}: R=Dilim_{w[0]:02d}, G=Dilim_{w[1]:02d}, B=Dilim_{w[2]:02d}')

## 🎓 EDA ÖZETİ VE ÇIKARILAN BÜYÜK DERSLER

| Analiz Alanı | Keşfedilen Kritik Gerçek | Modelleme Kararımız |
| :--- | :--- | :--- |
| **Düzlem Tamlığı** | Hastaların %100'ünde Sagittal + Coronal + Axial vardır. | Eksik veri sorunu yok; 3 düzlemi birleştiren Multi-View mimari kurulacak. |
| **Klinik Sinerji** | ACL yırtığı Kemik Ezilmesini (Contusion) 3 katına çıkarırken, Kireçlenmeyi (OA) düşürür. | Hastalıklar arası korelasyonu yakalayan Multi-Task / Shared Backbone modeli kullanılacak. |
| **Rapor Dilleri** | Raporlar tek bir dilde değil; İngilizce (%45), Fransızca (%25), İspanyolca (%15), Almanca (%15). | Kural/regex kesinlikle kullanılmayacak; Multilingual LLM / E5 etiket çıkarımı yapılacak. |
| **Görüntüleme** | 3 ardışık dilim yerel 3D doku değişimini renk kanalı gibi taşır. | 2.5D CoAtNet-384 + 12-Head Attention Pooling omurgası kurulacak. |